In [30]:
import collections

from Bio import pairwise2
from Bio.Seq import Seq
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd

from data import AllBooks, Book
from utils import most_frequent

from align_config import dss_version, sp_version
from align_functions import make_alignments, collect_matching_words

from tf.app import use

MT = use('etcbc/bhsa')
MT.load(['g_lex'])
Fmt, Tmt, Lmt = MT.api.F, MT.api.T, MT.api.L

SP = use('dt-ucph/sp:clone', checkout='clone', version=sp_version, provenanceSpec=dict(moduleSpecs=[]))
Fsp, Tsp, Lsp = SP.api.F, SP.api.T, SP.api.L

DSS = use('etcbc/dss', version=dss_version)
Fdss, Tdss, Ldss = DSS.api.F, DSS.api.T, DSS.api.L


**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,39,10938.21,100
chapter,929,459.19,100
lex,9230,46.22,100
verse,23213,18.38,100
half_verse,45179,9.44,100
sentence,63717,6.70,100
sentence_atom,64514,6.61,100
clause,88131,4.84,100
clause_atom,90704,4.70,100
phrase,253203,1.68,100


**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,5,79878.20,100
chapter,187,2135.78,100
verse,5841,68.38,100
word,114890,3.48,100
sign,399391,1.00,100


**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
scroll,1001,1428.81,100
lex,10450,129.14,94
fragment,11182,127.91,100
line,52895,27.04,100
clause,125,12.85,0
cluster,101099,6.68,47
phrase,315,5.10,0
word,500995,2.81,99
sign,1430241,1.00,100


In [97]:
SYLLABLE_TYPE = 'first'

MATRES_FILE = '../data/nouns_adjectives.csv'

In [98]:
PENTATEUCH_BOOKS = ['Genesis', 'Exodus', 'Leviticus', 'Numbers', 'Deuteronomy']
ALL_BOOK_NAMES = [Tmt.sectionFromNode(bo)[0] for bo in Fmt.otype.s('book')]

In [99]:
QSP_SCROLLS = {'1Qisaa', '1QisaaI', '1QisaaII', '2Q3', '4Q13', '4Q20', '2Q7', '4Q27', '1Q4', '2Q12', '4Q37', '4Q38', '4Q38a', '4Q40', '4Q53',
               '4Q57', '2Q13', '4Q78', '4Q80', '4Q82', '4Q128', '4Q129', '4Q134', '4Q135', '4Q136',
                '4Q137', '4Q138', '4Q139', '4Q140', '4Q141', '4Q142', '4Q143', '4Q144', '4Q158', '4Q364',
                '4Q365', '4Q96', '4Q111', '4Q109', '11Q5', '11Q6', '11Q7', '11Q8'}

# Prepare MT and SP texts

Produce dictionary mt_sp_matches which has mt words nodes as keys and matching word numbers from SP as values.

In [100]:
# prepare mt and sp books
MANUSCRIPTS = ['MT', 'SP']
all_books = AllBooks()
for book_name in ALL_BOOK_NAMES:
    book = Book('MT', book_name, Fmt, Tmt, Lmt)
    all_books.data[('MT', book_name)] = book
    
    if book_name in PENTATEUCH_BOOKS:
        book = Book('SP', book_name, Fsp, Tsp, Lsp)
        all_books.data[('SP', book_name)] = book

# Match words

In [101]:
dat = pd.read_csv(MATRES_FILE, sep='\t')
dat = dat[(dat.type == SYLLABLE_TYPE)]
print(dat.shape)

(10758, 37)


C:\Users\geitb\AppData\Local\Temp\ipykernel_29740\1368695180.py:1: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  dat = pd.read_csv(MATRES_FILE, sep='\t')


In [102]:
dat['lex_type'] = dat['lex'] + '_' + dat['type']
lex_types = set(dat.lex_type)

lex_types_dict = {}
for lt in lex_types:
    lt_df = dat[dat.lex_type == lt]
    letter_count = collections.Counter(lt_df.vowel_letter)
    letter_count2 = collections.Counter({k:v for k, v in letter_count.items() if isinstance(k, str)})
    most = letter_count2.most_common(1)[0][0]
    lex_types_dict[lt] = most

dat['vowel_type'] = [lex_types_dict[lex_typ] for lex_typ in dat.lex_type]

In [103]:
dat = dat[dat.vowel_type == 'W']

In [104]:
dat.head()

,tf_id,scroll,book,chapter,verse,lex,g_cons,stem,pattern,pattern_g_cons,...,has_hloc,has_nme,rec_signs_stem,cor_signs_stem,type,vowel_letter,has_vowel_letter,neigh_vowel_letter,lex_type,vowel_type
3,20,MT,Genesis,1,2,XCK/,XCK,XCK,CCC,CCC,...,0,0,nnn,nnn,first,NaN,0,0,XCK/_first,W
9,57,MT,Genesis,1,4,XCK/,XCK,XCK,CCC,CCC,...,0,0,nnn,nnn,first,NaN,0,0,XCK/_first,W
11,68,MT,Genesis,1,5,XCK/,XCK,XCK,CCC,CCC,...,0,0,nnn,nnn,first,NaN,0,0,XCK/_first,W
12,76,MT,Genesis,1,5,BQR=/,BQR,BQR,CCC,CCC,...,0,0,nnn,nnn,first,NaN,0,0,BQR=/_first,W
16,137,MT,Genesis,1,8,BQR=/,BQR,BQR,CCC,CCC,...,0,0,nnn,nnn,first,NaN,0,0,BQR=/_first,W


In [105]:
# dat_dss means non-MT, SP is included
dat_dss = dat[~dat.scroll.isin(['MT'])]
scroll_book_combinations = list(set(zip(dat_dss.scroll, dat_dss.book)))
dat_dss.shape

(3184, 39)

In [106]:
dat
dat_mt = dat[dat.scroll.isin(['MT'])]
dat_mt.shape

(6534, 39)

In [107]:
dat_mt_sg = dat_mt[dat_mt.nu == 'sg']
dat_mt_pl = dat_mt[dat_mt.nu == 'pl']

In [108]:
g_lexs = ''.join({Fmt.g_lex.v(w) for w in dat_mt_sg.tf_id})
set(g_lexs)

{'.',
 ':',
 ';',
 '<',
 '>',
 '@',
 'A',
 'B',
 'C',
 'D',
 'E',
 'F',
 'G',
 'H',
 'I',
 'J',
 'K',
 'L',
 'M',
 'N',
 'O',
 'P',
 'Q',
 'R',
 'S',
 'T',
 'U',
 'V',
 'W',
 'X',
 'Y',
 'Z'}

In [109]:
lex_no_ou = []

lexemes = set(dat_mt_pl.lex)
for lex in lexemes:
    dat_pl_lex = dat_mt_pl[dat_mt_pl.lex == lex]
    g_lexs = ''.join({Fmt.g_lex.v(w) for w in dat_pl_lex.tf_id})
    if not 'O' in g_lexs and not 'U' in g_lexs and not 'W.' in g_lexs:
        print(lex)
        lex_no_ou.append(lex)

XRB/
RKS/
XQ/
QDC/
P<L/
XRC/
QBR/
NKRJ/
CRC/
DDNJ/
BQR=/
XV>=/
LHBH/
XRBH=/
GPN/
RKB/
CQMH/
<CN=/
JBL/
RG</
KRKRH/
<RLH/
PCT/
XDC=/
XPCJ/
C<L/


In [110]:
dat_dss_pl_no_ou = dat_dss[(dat_dss.lex.isin(lex_no_ou)) & (dat_dss.nu == 'pl')]

In [111]:
pd.crosstab(dat_dss_pl_no_ou.lex, dat_dss_pl_no_ou.has_vowel_letter)

has_vowel_letter,0,1
lex,,
<CN=/,0,1
<RLH/,1,0
BQR=/,2,0
CQMH/,2,2
CRC/,1,1
GPN/,1,0
JBL/,0,2
KRKRH/,1,1
LHBH/,0,1


In [114]:
dat_dss_pl_no_ou[dat_dss_pl_no_ou.lex == 'XDC=/']

,tf_id,scroll,book,chapter,verse,lex,g_cons,stem,pattern,pattern_g_cons,...,has_hloc,has_nme,rec_signs_stem,cor_signs_stem,type,vowel_letter,has_vowel_letter,neigh_vowel_letter,lex_type,vowel_type
37196,526855,SP,Genesis,38,24,XDC=/,XDCJM,XDC,CCC,CCCMC,...,0,1,nnn,nnn,first,NaN,0,0,XDC=/_first,W
38318,541056,SP,Exodus,12,2,XDC=/,XDCJM,XDC,CCC,CCCMC,...,0,1,nnn,nnn,first,NaN,0,0,XDC=/_first,W
38319,541061,SP,Exodus,12,2,XDC=/,XDCJ,XDC,CCC,CCCM,...,0,1,nnn,nnn,first,NaN,0,0,XDC=/_first,W
42633,583056,SP,Numbers,10,10,XDC=/,XDCJKM,XDC,CCC,CCCMCC,...,0,1,nnn,nnn,first,NaN,0,0,XDC=/_first,W
43439,594837,SP,Numbers,28,11,XDC=/,XDCJKM,XDC,CCC,CCCMCC,...,0,1,nnn,nnn,first,NaN,0,0,XDC=/_first,W
43454,594928,SP,Numbers,28,14,XDC=/,XDCJ,XDC,CCC,CCCM,...,0,1,nnn,nnn,first,NaN,0,0,XDC=/_first,W
45695,1895093,1Qisaa,Isaiah,1,14,XDC=/,XWDCJKM,XWDC,CMCC,CMCCMCC,...,0,1,nnnn,nnnn,first,W,1,0,XDC=/_first,W
48589,1942917,4Q11,Exodus,12,2,XDC=/,XDCJM,XDC,CCC,CCCMC,...,0,1,nnn,nnn,first,NaN,0,0,XDC=/_first,W
48590,1942922,4Q11,Exodus,12,2,XDC=/,XDCJ,XDC,CCC,CCCM,...,0,1,nnn,nnn,first,NaN,0,0,XDC=/_first,W
48854,1954439,4Q22,Exodus,12,2,XDC=/,XDCJ,XDC,CCC,CCCM,...,0,1,nnn,nnn,first,NaN,0,0,XDC=/_first,W


In [79]:
dat_dss_pl_no_ou[dat_dss_pl_no_ou.lex == 'YPWR/']

,tf_id,scroll,book,chapter,verse,lex,g_cons,stem,pattern,pattern_g_cons,...,has_hloc,has_nme,rec_signs_stem,cor_signs_stem,type,vowel_letter,has_vowel_letter,neigh_vowel_letter,lex_type,vowel_type
36259,512373,SP,Genesis,15,10,YPWR/,YPWRJM,YPWR,CCMC,CCMCMC,...,0,1,nnnn,nnnn,last,W,1,0,YPWR/_last,W
41072,567058,SP,Leviticus,14,4,YPWR/,YPWRJM,YPWR,CCMC,CCMCMC,...,0,1,nnnn,nnnn,last,W,1,0,YPWR/_last,W
41200,568106,SP,Leviticus,14,49,YPWR/,YPWRJM,YPWR,CCMC,CCMCMC,...,0,1,nnnn,nnnn,last,W,1,0,YPWR/_last,W
46597,1905805,1Qisaa,Isaiah,31,5,YPWR/,YPWRJM,YPWR,CCMC,CCMCMC,...,0,1,nnnn,nnnn,last,W,1,0,YPWR/_last,W
50492,2043053,4Q82,Hosea,11,11,YPWR/,YPRJM,YPR,CCC,CCCMC,...,0,1,nnn,nnn,last,NaN,0,0,YPWR/_last,W


In [113]:
dat[dat.lex == 'JBL/']

,tf_id,scroll,book,chapter,verse,lex,g_cons,stem,pattern,pattern_g_cons,...,has_hloc,has_nme,rec_signs_stem,cor_signs_stem,type,vowel_letter,has_vowel_letter,neigh_vowel_letter,lex_type,vowel_type
16939,222278,MT,Isaiah,30,25,JBL/,JBLJ,JBL,CCC,CCCM,...,0,1,nnn,nnn,first,NaN,0,0,JBL/_first,W
17414,227402,MT,Isaiah,44,4,JBL/,JBLJ,JBL,CCC,CCCM,...,0,1,nnn,nnn,first,NaN,0,0,JBL/_first,W
46570,1905484,1Qisaa,Isaiah,30,25,JBL/,JWBLJ,JWBL,CMCC,CMCCM,...,0,1,nnnn,nnnn,first,W,1,0,JBL/_first,W
47053,1910955,1Qisaa,Isaiah,44,4,JBL/,JWBLJ,JWBL,CMCC,CMCCM,...,0,1,nnnn,nnnn,first,W,1,0,JBL/_first,W
